In [0]:
%run
"/Workspace/Jurídico/Normalizador_de_dados/Normalizador"

In [0]:
#Cria o Widget para seleção do mês por extenso
meses_do_ano = [
    "JANEIRO", "FEVEREIRO", "MARÇO", "ABRIL", "MAIO", "JUNHO",
    "JULHO", "AGOSTO", "SETEMBRO", "OUTUBRO", "NOVEMBRO", "DEZEMBRO"
]

dbutils.widgets.dropdown(
    name="Mes_Extenso", 
    defaultValue="JANEIRO", 
    choices=meses_do_ano, 
    label="Selecione o Mês por Extenso"
)

Meses = dbutils.widgets.get("Mes_Extenso")

In [0]:
#CAMINHOS DOS DIRETORIOS 

caminho_arquivos_individual = f'/Volumes/databox/juridico_comum/arquivos/financeiro/Faturamento_Pericias/Base_Mensal/FATURAMENTO PERICIAS {Meses}.xlsx'
caminho_arquivo_referencia = f'/Volumes/databox/juridico_comum/arquivos/financeiro/Faturamento_Pericias/Referencia_Precificacao/REFERENCIA.xlsb'
caminho_arquivo_precificao_pericia = f'/Volumes/databox/juridico_comum/arquivos/financeiro/Faturamento_Pericias/Referencia_Precificacao/Tabela de Precificação Calculistas e Peritos - Versão Atualizada 3.xlsx'
caminho_arquivo_historico = f'/Volumes/databox/juridico_comum/arquivos/financeiro/Faturamento_Pericias/Arquivo_Consolidado/Consolidado Faturamento - Calculos e Perícias 2023 e 2025.xlsx'
DIRETORIO_SAIDA = f'/Volumes/databox/juridico_comum/arquivos/financeiro/Faturamento_Pericias/Faturamento_Final/'


df_pericia = pd.read_excel(
            caminho_arquivos_individual,
            sheet_name="PERÍCIAS TAREFAS - FATURAMENTO",
            header=5,
            engine="openpyxl")

df_precificacao_pericias_METRA = pd.read_excel(
            caminho_arquivo_precificao_pericia,
            sheet_name="PRECIFICAÇÃO PERÍCIAS",
            header=15,
            nrows = 5,
            engine="openpyxl")

df_precificacao_pericias_VENDRAME = pd.read_excel(
            caminho_arquivo_precificao_pericia,
            sheet_name="PRECIFICAÇÃO PERÍCIAS",
            header= 7,
            nrows = 5,
            engine="openpyxl")

df_precificacao_pericias_RHUMAS = pd.read_excel(
            caminho_arquivo_precificao_pericia,
            sheet_name="PRECIFICAÇÃO PERÍCIAS",
            header=1,
            nrows = 2,
            engine="openpyxl")


df_precificao_geral_descontos = pd.read_excel(
            caminho_arquivo_precificao_pericia,
            sheet_name="PRECIFICAÇÃO PERÍCIAS",
            usecols="A,B",
            header=24,
            engine="openpyxl")

df_referencia = pd.read_excel(
            caminho_arquivo_referencia,
            usecols="F,G",
            sheet_name="Planilha1",
            header=0,
            engine="pyxlsb")

#df_historico = pd.read_excel(
  #          caminho_arquivo_historico,
   #         sheet_name="PERÍCIAS",
    #        header=1,
     #       engine="openpyxl")

In [0]:
# Correto: Passando a chave e o valor
df_historico_spark = spark.table("databox.juridico_comum.br_consolidado_pericias")
df_historico = df_historico_spark.toPandas()


In [0]:
df_historico.head()

**LIMPA COLUNAS COM ESPAÇOS NO INÍCIO E NO FIM**

In [0]:
colunas_limpa = []

for col in df_pericia.columns:
    # Converte para string (str) antes de dar o strip
    colunas_limpa.append(str(col).strip()) 

df_pericia.columns = colunas_limpa

**RENOMEIA COLUNAS REPETIDAS COM _1, _2, ETC.**

In [0]:
contador = {}
novos_nomes = []

for col in df_pericia.columns:
    if col in contador:
        contador[col] += 1
        novo_nome = f"{col}_{contador[col]}"
    else:
        contador[col] = 0
        novo_nome = col
    novos_nomes.append(novo_nome)

df_pericia.columns = novos_nomes

**NOME E ORDEM DAS COLUNAS DESEJADAS**

In [0]:
print(df_pericia.columns.tolist())

In [0]:
colunas_desejadas = ['PROCESSO - ID', 
                     'TAREFA - ID', 
                     'NÚMERO DO PROCESSO',
                     'EMPRESA A DEFINIR ASSISTENTE TÉCNICO',
                     'ESCRITÓRIO',
                     'ADVOGADO RESPONSÁVEL',
                     'STATUS',
                     'TIPO DA TAREFA',
                     'FICHAS DE PROCESSO - ID',
                     'PRAZO (SLA)',
                     'DATA DE CONFIRMAÇÃO',
                     'STATUS DA TAREFA',
                	 'RESPONSÁVEL PELA CONFIRMAÇÃO',
                     'TIPO DE PERÍCIA',
                     'ANEXAR RESPOSTA À IMPUGNAÇÃO?',
                     'ESCRITÓRIO_1',
                	 'ANÁLISE'
]

df_selecionado = df_pericia[colunas_desejadas]

**VALIDAR COLUNAS PARA FATURAR**

In [0]:
condicao_faturar = df_selecionado['ANÁLISE'].str.contains('FATURAR', case=False, na=False)
df_selecionado = df_selecionado[condicao_faturar].copy()

df_selecionado.loc[
    df_selecionado['ANÁLISE'].str.contains('NÃO FATURAR', case=False, na=False), 
    'FATURAR?'
] = 'NÃO FATURADO'

In [0]:
df_selecionado.loc[
    df_selecionado['FATURAR?'].isnull(), 
    'FATURAR?'
] = 'FATURADO'

In [0]:
df_selecionado.loc[df_selecionado['ANÁLISE'].str.contains('NÃO FATURAR', case=False, na=False), 'FATURAR?'] = 'NÃO FATURADO'

**MERGE COM PARA AJUSTE DA COLUNA DE PERÍCIA**

In [0]:
df_selecionado = df_selecionado.merge(
     df_referencia,
     left_on=['TIPO DE PERÍCIA'],
     right_on=['DE TIPO DE PERÍCIA'],
     how='left'
 )

In [0]:
df_selecionado = df_selecionado.drop(columns=['DE TIPO DE PERÍCIA'])

In [0]:
df_selecionado = df_selecionado.rename(columns={'PARA TIPO DE PERÍCIA': 'PERÍCIA AJUSTADA'})

**VERIFICANDO QTDS DE PERÍCIA POR TIPO E ESCRITÓRIO**

In [0]:
tabela_qtd_pericias_METRA = df_selecionado.groupby(['ESCRITÓRIO_1', 'PERÍCIA AJUSTADA']).size().reset_index(name='QTD')

In [0]:
print(tabela_qtd_pericias_METRA)

**TRANSFORMANDO TABELA DE REFERÊNCIA PARA PRECIFICAÇÃO**

In [0]:
df_long = df_precificacao_pericias_METRA.melt(
    id_vars=["ESCRITÓRIO","TIPO DE PERÍCIA", "TIPO ACAO"],
    var_name="FAIXA DE PERÍCIAS",
    value_name="VALOR"
)

In [0]:
bins = [0, 10, 30, 50, float('inf')]
labels = [
    'ATÉ 10 PERÍCIAS',
    'ENTRE 11 E 30 PERÍCIAS',
    'ENTRE 31 E 50 PERÍCIAS',
    'ACIMA DE 50 PERÍCIAS'
]

In [0]:
tabela_qtd_pericias_METRA['FAIXA_PERICIAS'] = pd.cut(tabela_qtd_pericias_METRA['QTD'], bins=bins, labels=labels, right=True)

In [0]:
tabela_qtd_pericias_METRA['CHAVE_VALIDACAO'] = tabela_qtd_pericias_METRA['ESCRITÓRIO_1'].astype(str) + '_' + tabela_qtd_pericias_METRA['PERÍCIA AJUSTADA'].astype(str) + '_' + tabela_qtd_pericias_METRA['FAIXA_PERICIAS'].astype(str)

**DESCOBRINDO O VALOR da PRECIFICAÇÃO O METRA*

In [0]:
df_long['CHAVE_QTD'] = df_long['ESCRITÓRIO'].astype(str) + '_' + df_long['TIPO DE PERÍCIA'].astype(str) + '_' + df_long['FAIXA DE PERÍCIAS'].astype(str)

In [0]:
tabela_qtd_pericias_METRA['CHAVE_QTD'] = tabela_qtd_pericias_METRA['ESCRITÓRIO_1'].astype(str) + '_' + tabela_qtd_pericias_METRA['PERÍCIA AJUSTADA'].astype(str) + '_' + tabela_qtd_pericias_METRA['FAIXA_PERICIAS'].astype(str)

In [0]:
tabela_qtd_pericias_METRA = tabela_qtd_pericias_METRA.merge(
     df_long,
     left_on=['CHAVE_QTD'],
     right_on=['CHAVE_QTD'],
     how='left'
 )

In [0]:
tabela_qtd_pericias_METRA = tabela_qtd_pericias_METRA.drop(columns=['CHAVE_QTD','ESCRITÓRIO','TIPO DE PERÍCIA','TIPO ACAO','FAIXA DE PERÍCIAS','FAIXA_PERICIAS','CHAVE_VALIDACAO'])

**MERGE COM DO DF SELECIONADO COM tabela_qtd_pericias_METRA PARA PRECIFICAR LINHA A LINHA**

In [0]:
tabela_qtd_pericias_METRA['CHAVE_ESCRITORIO_METRA'] = tabela_qtd_pericias_METRA['ESCRITÓRIO_1'].astype(str) + '_' + tabela_qtd_pericias_METRA['PERÍCIA AJUSTADA'].astype(str)

In [0]:
df_selecionado['CHAVE_ESCRITORIO_METRA'] = df_selecionado['ESCRITÓRIO_1'].astype(str) + '_' + df_selecionado['PERÍCIA AJUSTADA'].astype(str)

In [0]:
df_selecionado['VALOR_METRA'] = df_selecionado['CHAVE_ESCRITORIO_METRA'].map(tabela_qtd_pericias_METRA.set_index('CHAVE_ESCRITORIO_METRA')['VALOR'])

In [0]:
df_selecionado = df_selecionado.drop(columns=['CHAVE_ESCRITORIO_METRA'])

**PRECIFICAÇÃO PARA O RHUMAS**

In [0]:
df_selecionado['CHAVE_ESCRITORIO_RHUMAS'] = df_selecionado['ESCRITÓRIO_1'].astype(str) + '_' + df_selecionado['TIPO DA TAREFA'].astype(str)

In [0]:
df_precificacao_pericias_RHUMAS['CHAVE_ESCRITORIO_RHUMAS_REF'] = df_precificacao_pericias_RHUMAS['ESCRITÓRIO'].astype(str) + '_' + df_precificacao_pericias_RHUMAS['TIPO DE CÁLCULO CONTRATO'].astype(str)

In [0]:
df_selecionado['VALOR_RHUMAS'] = df_selecionado['CHAVE_ESCRITORIO_RHUMAS'].map(df_precificacao_pericias_RHUMAS.set_index('CHAVE_ESCRITORIO_RHUMAS_REF')['VLR ÚNICO'])

In [0]:
df_selecionado = df_selecionado.drop(columns=['CHAVE_ESCRITORIO_RHUMAS'])

**PRECIFICAÇÃO PARA O VENDRAME**

In [0]:
df_selecionado['CHAVE_ESCRITORIO_VENDRAME'] = df_selecionado['ESCRITÓRIO_1'].astype(str) + '_' + df_selecionado['PERÍCIA AJUSTADA'].astype(str)

In [0]:
df_precificacao_pericias_VENDRAME['CHAVE_ESCRITORIO_VENDRAME'] = df_precificacao_pericias_VENDRAME['ESCRITÓRIO'].astype(str) + '_' + df_precificacao_pericias_VENDRAME['TIPO DE PERÍCIA'].astype(str)

In [0]:
df_selecionado['VALOR_VENDRAME'] = df_selecionado['CHAVE_ESCRITORIO_VENDRAME'].map(df_precificacao_pericias_VENDRAME.set_index('CHAVE_ESCRITORIO_VENDRAME')['VLR ÚNICO PERÍCIA'])

In [0]:
df_selecionado = df_selecionado.drop(columns=['CHAVE_ESCRITORIO_VENDRAME'])

**UNIFICANDO COLUNA DE VALORES**

In [0]:
df_selecionado['VALOR'] = df_selecionado[['VALOR_METRA', 'VALOR_RHUMAS', 'VALOR_VENDRAME']].bfill(axis=1).iloc[:, 0]

In [0]:
df_selecionado = df_selecionado.drop(columns=['VALOR_METRA','VALOR_RHUMAS','VALOR_VENDRAME'])

In [0]:
df_selecionado.loc[df_selecionado['FATURAR?'] == 'NÃO FATURADO', 'VALOR'] = 0

**APLICANDO DESCONTOS PARA VENDRAME E METRA**

In [0]:
# Crie o dicionário de busca (estilo PROCV)
dicionario = df_precificao_geral_descontos.set_index('QUESITO')['VALOR']

# Aplique a condição antes de mapear
df_selecionado['DESCONTO'] = np.where(
    ((df_selecionado['TIPO DA TAREFA'] == 'ANEXAR QUESITO') | (df_selecionado['TIPO DA TAREFA'] == 'ANEXAR IMPUGNAÇÃO/MANIFESTAÇÃO')) & ((df_selecionado['ESCRITÓRIO_1'] != 'RUHMAS') & (df_selecionado['ESCRITÓRIO_1'] != 'BAPTISTA E SOUZA')),
    df_selecionado['TIPO DA TAREFA'].map(dicionario),
    np.nan
)

In [0]:
df_selecionado.loc[df_selecionado['VALOR'] == "", 'DESCONTO'] = 0

In [0]:
df_selecionado['VALOR_FINAL'] = np.where(
    df_selecionado['DESCONTO'].notna(),
    df_selecionado['VALOR'] * df_selecionado['DESCONTO'],
    df_selecionado['VALOR']
)

**VALIDANDO SE JÁ HOUVE PAGAMENTO**

In [0]:
df_selecionado['PROCESSO - ID'] = df_selecionado['PROCESSO - ID'].fillna(0).astype(int)
df_selecionado['TAREFA - ID'] = df_selecionado['TAREFA - ID'].fillna(0).astype(int)

In [0]:
df_historico.head()

In [0]:
df_historico['TAREFA_ID'] = df_historico['TAREFA_ID'].astype(str)
df_historico['TAREFA_ID'] = df_historico['TAREFA_ID'].str.replace('_x000D_', '', regex=False).str.strip()
# Isso garante que se sobrar algum lixo irreconhecível, ele vira NaN (nulo) em vez de travar o código
df_historico['TAREFA_ID'] = pd.to_numeric(df_historico['TAREFA_ID'], errors='coerce')

In [0]:
df_historico['PROCESSO_ID'] = df_historico['PROCESSO_ID'].fillna(0).astype(int)
df_historico['TAREFA_ID'] = df_historico['TAREFA_ID'].fillna(0).astype(int)

In [0]:
df_historico['CHAVE_VALIDACAO_HIST'] =  df_historico['PROCESSO_ID'].astype(str) \
                                + '_' + df_historico['TIPO_DE_PERICIA'].astype(str) \
                                + '_' + df_historico['FATURAR'].astype(str) \
                                + '_' + df_historico['FORNECEDOR'].astype(str) \
                                + '_' + df_historico['TIPO_DA_TAREFA'].astype(str) 

In [0]:
df_selecionado['CHAVE_VALIDACAO_ATUAL'] =   df_selecionado['PROCESSO - ID'].astype(str) \
                                    + '_' + df_selecionado['PERÍCIA AJUSTADA'].astype(str) \
                                    + '_' + df_selecionado['FATURAR?'].astype(str) \
                                    + '_' + df_selecionado['ESCRITÓRIO_1'].astype(str) \
                                    + '_' + df_selecionado['TIPO DA TAREFA'].astype(str) 

In [0]:
df_selecionado['HOUVE PAGAMENTO EM MES ANTERIOR'] = df_selecionado['CHAVE_VALIDACAO_ATUAL'].isin(
    df_historico['CHAVE_VALIDACAO_HIST']
)

In [0]:
contagem_chaves = df_historico['CHAVE_VALIDACAO_HIST'].value_counts()

In [0]:
df_selecionado['QTD_OCORRENCIAS_NO_HISTORICO'] = df_selecionado['CHAVE_VALIDACAO_ATUAL'].map(contagem_chaves)

In [0]:
df_selecionado['QTD_OCORRENCIAS_NO_HISTORICO'] = (
    df_selecionado['QTD_OCORRENCIAS_NO_HISTORICO'].fillna(0).astype(int)
)

In [0]:
soma_pagamentos = df_historico.groupby('CHAVE_VALIDACAO_HIST')['PRECO_FINAL'].sum()

In [0]:
df_selecionado['VALOR_PAGAMENTOS_ANTERIORES'] = df_selecionado['CHAVE_VALIDACAO_ATUAL'].map(soma_pagamentos)

In [0]:
df_selecionado['VALOR_PAGAMENTOS_ANTERIORES'] = df_selecionado['VALOR_PAGAMENTOS_ANTERIORES'].fillna(0)

In [0]:
df_selecionado['QTD_OCORRENCIAS_NO_HISTORICO'] = df_selecionado['QTD_OCORRENCIAS_NO_HISTORICO'].fillna(0).astype(int)

In [0]:
df_selecionado.loc[df_selecionado['QTD_OCORRENCIAS_NO_HISTORICO'] == 0, 'RESOLVIDO?'] = 'RESOLVIDO'

In [0]:
df_selecionado.loc[df_selecionado['QTD_OCORRENCIAS_NO_HISTORICO'] >= 1, 'FATURAR?'] = 'NÃO FATURAR'
df_selecionado.loc[df_selecionado['QTD_OCORRENCIAS_NO_HISTORICO'] >= 1, 'RESOLVIDO?'] = 'RESOLVIDO'
df_selecionado.loc[df_selecionado['QTD_OCORRENCIAS_NO_HISTORICO'] >= 1, 'ANÁLISE'] = 'NÃO FATURAR - PAGO EM MÊS ANTERIOR'

In [0]:
import os
import re
import shutil
import time # Importante para dar um respiro ao sistema de arquivos

# --- CONFIGURAÇÕES ---
coluna_base = 'ESCRITÓRIO_1'
valores_unicos = df_selecionado[coluna_base].dropna().unique()

# Diretórios
DIRETORIO_FINAL = '/Volumes/databox/juridico_comum/arquivos/financeiro/Faturamento_Pericias/Faturamento_Final/'
DIRETORIO_TEMP = '/tmp/temp_faturamento_pericias'

# --- 1. PREPARAÇÃO (Reset dos diretórios) ---
print(f"🔧 Preparando diretórios...")

os.makedirs(DIRETORIO_FINAL, exist_ok=True)

if os.path.exists(DIRETORIO_TEMP):
    shutil.rmtree(DIRETORIO_TEMP) # Limpa lixo anterior
os.makedirs(DIRETORIO_TEMP, exist_ok=True)

print("🚀 Iniciando salvamento...")

for valor in valores_unicos:
    df_filtrado = df_selecionado[df_selecionado[coluna_base] == valor]

    # --- Tratamento do nome ---
    nome_seguro = str(valor).strip()
    nome_seguro = re.sub(r'[\\/*?:"<>|]', '', nome_seguro)
    nome_seguro = nome_seguro.replace(' ', '_').replace('/', '-')
    
    nome_arquivo = f"{coluna_base}_{nome_seguro}.xlsx"
    
    # Caminhos
    caminho_local = os.path.join(DIRETORIO_TEMP, nome_arquivo)
    caminho_volume = os.path.join(DIRETORIO_FINAL, nome_arquivo)

    try:
        # 1. Salva no disco local (/tmp)
        df_filtrado.to_excel(caminho_local, index=False)
        
        # 2. Verifica se já existe no destino e remove (evita conflito de sobrescrita)
        if os.path.exists(caminho_volume):
            try:
                os.remove(caminho_volume)
            except OSError:
                pass # Se não der pra apagar, o copyfile tenta sobrescrever mesmo assim

        # 3. Copia APENAS O CONTEÚDO (copyfile em vez de copy2)
        # copyfile é mais robusto para Volumes de nuvem pois ignora metadados de data/hora
        shutil.copyfile(caminho_local, caminho_volume)
        
        # 4. Remove o temporário
        os.remove(caminho_local)
        
        print(f"✅ Sucesso: {nome_arquivo}")
        
    except Exception as e:
        print(f"❌ Erro crítico ao processar {nome_arquivo}: {e}")
        # Tenta usar o comando nativo do Databricks como 'Plano C' se o Python falhar
        try:
            print(f"   ...Tentando método alternativo (dbutils) para {nome_arquivo}...")
            # Recria o arquivo local caso tenha sido apagado
            df_filtrado.to_excel(caminho_local, index=False)
            dbutils.fs.cp(f"file:{caminho_local}", caminho_volume.replace("/Volumes", "dbfs:/Volumes"))
            print(f"   ✅ Sucesso via dbutils!")
        except Exception as e2:
             print(f"   ☠️ Falha total no arquivo {nome_arquivo}: {e2}")

    # Pequena pausa para o sistema de arquivos processar
    time.sleep(0.5)

# Limpeza final
try:
    shutil.rmtree(DIRETORIO_TEMP)
except:
    pass

print("🏁 Processo finalizado.")